In [ ]:
import sys
from pathlib import Path
import os

# Ensure project root and src are in Python path
PROJECT_ROOT = Path.cwd().parent  # Go from research/ to parent (project root)
SRC_PATH = PROJECT_ROOT / "src"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))
    
# Change working directory to project root
os.chdir(PROJECT_ROOT)

print(f"Project root: {PROJECT_ROOT}")
print(f"Working directory: {os.getcwd()}")

In [ ]:
# Force reload textsummarizer modules
if 'textsummarizer' in sys.modules:
    del sys.modules['textsummarizer']
if 'textsummarizer.constants' in sys.modules:
    del sys.modules['textsummarizer.constants']
if 'textsummarizer.utils' in sys.modules:
    del sys.modules['textsummarizer.utils']
if 'textsummarizer.logging' in sys.modules:
    del sys.modules['textsummarizer.logging']


'c:\\Users\\jayan\\Desktop\\text-summarization-project\\research'

In [3]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class DataValidationConfig:
    root_dir: Path
    STATUS_FILE: str
    ALL_REQUIRED_FILES: list

In [5]:
from textsummarizer.constants import *
from textsummarizer.utils.common import read_yaml, create_directories

In [10]:
from textsummarizer.constants import PROJECT_ROOT

class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])


    
    def get_data_validation_config(self) -> DataValidationConfig:
        config = self.config.data_validation

        # Convert relative paths to absolute paths
        root_dir = Path(config.root_dir) if Path(config.root_dir).is_absolute() else PROJECT_ROOT / config.root_dir
        status_file = Path(config.STATUS_FILE) if Path(config.STATUS_FILE).is_absolute() else PROJECT_ROOT / config.STATUS_FILE

        create_directories([root_dir])
        create_directories([status_file.parent])  # Ensure status file directory exists

        data_validation_config = DataValidationConfig(
            root_dir=root_dir,
            STATUS_FILE=str(status_file),
            ALL_REQUIRED_FILES=config.ALL_REQUIRED_FILES,
        )

        return data_validation_config


In [7]:
import os
from textsummarizer.logging import logger

In [11]:
import os
from textsummarizer.logging import logger

class DataValidation:
    def __init__(self, config: DataValidationConfig):
        self.config = config


    
    def validate_all_files_exist(self) -> bool:
        try:
            validation_status = None
            
            # Use absolute path to the dataset
            dataset_path = PROJECT_ROOT / "artifacts" / "data_ingestion" / "samsum_dataset"
            
            if not dataset_path.exists():
                validation_status = False
                logger.error(f"Dataset directory not found: {dataset_path}")
                with open(self.config.STATUS_FILE, 'w') as f:
                    f.write(f"Validation status: {validation_status}")
                return validation_status

            all_files = os.listdir(dataset_path)
            logger.info(f"Found files: {all_files}")
            logger.info(f"Required files: {self.config.ALL_REQUIRED_FILES}")

            for file in all_files:
                if file not in self.config.ALL_REQUIRED_FILES:
                    validation_status = False
                    logger.warning(f"File {file} is not in required files")
                    with open(self.config.STATUS_FILE, 'w') as f:
                        f.write(f"Validation status: {validation_status}")
                else:
                    validation_status = True
                    with open(self.config.STATUS_FILE, 'w') as f:
                        f.write(f"Validation status: {validation_status}")

            logger.info(f"Validation completed. Status: {validation_status}")
            return validation_status
        
        except Exception as e:
            logger.error(f"Validation error: {str(e)}")
            raise e


In [12]:
try:
    config = ConfigurationManager()
    data_validation_config = config.get_data_validation_config()
    print(f"Validation config root_dir: {data_validation_config.root_dir}")
    print(f"Status file: {data_validation_config.STATUS_FILE}")
    print(f"Required files: {data_validation_config.ALL_REQUIRED_FILES}")
    
    data_validation = DataValidation(config=data_validation_config)
    result = data_validation.validate_all_files_exist()
    print(f"✅ Validation completed successfully! Result: {result}")
except Exception as e:
    print(f"❌ Error: {type(e).__name__}: {str(e)}")
    import traceback
    traceback.print_exc()


[2026-04-06 10:57:37,973: INFO: common: yaml file: C:\Users\jayan\Desktop\text-summarization-project\config\config.yaml loaded successfully]
[2026-04-06 10:57:37,980: INFO: common: yaml file: C:\Users\jayan\Desktop\text-summarization-project\params.yaml loaded successfully]
[2026-04-06 10:57:37,982: INFO: common: created directory at: artifacts]
[2026-04-06 10:57:37,985: INFO: common: created directory at: C:\Users\jayan\Desktop\text-summarization-project\data_validation]
[2026-04-06 10:57:37,989: INFO: common: created directory at: C:\Users\jayan\Desktop\text-summarization-project\artifacts\data_validation]
Validation config root_dir: C:\Users\jayan\Desktop\text-summarization-project\data_validation
Status file: C:\Users\jayan\Desktop\text-summarization-project\artifacts\data_validation\status.txt
Required files: ['train', 'test', 'validation']
[2026-04-06 10:57:37,996: INFO: 3230525561: Found files: ['dataset_dict.json', 'test', 'train', 'validation']]
[2026-04-06 10:57:37,997: INFO: